In [24]:
# Performing 70-30 train-validation split within the entire region
# Stations within 1 pixel distance (diagonal; 15.74km) of ERA5L (coarsest data) is avoided from split using DBSCAN
# added clusterwise split

In [25]:
import pandas as pd
import numpy as np
import rasterio as rio
import os
from sklearn.cluster import DBSCAN

output_dir = r"..\Data_Temp\Processed\Split_stations"
os.makedirs(output_dir, exist_ok= True)

temp_cluster_path = r"..\Data_Temp\Processed\Spatial_clustering\Precipitation_clusters.tif"

tmax_insitu_df = pd.read_parquet(r"..\Data_Temp\Insitu\Cleaned\tmax_cleaned.parquet")
tmin_insitu_df = pd.read_parquet(r"..\Data_Temp\Insitu\Cleaned\tmin_cleaned.parquet")

kms_per_radian = 6371.0088
search_distance_km = 16
epsilon = search_distance_km / kms_per_radian
min_samples_per_cluster = 2

In [26]:
# create a df of unique station values
tmax_stn_df = tmax_insitu_df.drop_duplicates(subset=['index_no'], keep='first').reset_index(drop=True)
tmin_stn_df = tmin_insitu_df.drop_duplicates(subset=['index_no'], keep='first').reset_index(drop=True)

tmax_stn_df = tmax_stn_df[['index_no', 'lat', 'long']]
tmin_stn_df = tmin_stn_df[['index_no', 'lat', 'long']]

# Sample stations to extract cluster values
tmax_stn_points = list(zip(tmax_stn_df['long'], tmax_stn_df['lat']))
tmin_stn_points = list(zip(tmin_stn_df['long'], tmin_stn_df['lat']))

#extract cluster value - tmax
tmax_cluster_values = []
with rio.open(temp_cluster_path) as src:
    tmax_sampled_values = list(src.sample(tmax_stn_points))
    for i, value in enumerate(tmax_sampled_values):
        tmax_cluster_values.append(int(value[0]))
         
#add cluster values to station df
tmax_stn_df['cluster_id'] = tmax_cluster_values

#extract cluster value - tmin
tmin_cluster_values = []
with rio.open(temp_cluster_path) as src:
    tmin_sampled_values = list(src.sample(tmin_stn_points))
    for i, value in enumerate(tmin_sampled_values):
        tmin_cluster_values.append(int(value[0]))
         
#add cluster values to station df
tmin_stn_df['cluster_id'] = tmin_cluster_values

In [27]:
# Assign DB Cluster value to each station

# For Tmax
tmax_coords = np.radians(tmax_stn_df[['lat', 'long']].to_numpy())
tmax_db = DBSCAN(eps = epsilon, min_samples=min_samples_per_cluster, algorithm='ball_tree', metric = 'haversine').fit(tmax_coords)
tmax_stn_df['db_cluster'] = tmax_db.labels_
tmax_noise_count = tmax_stn_df[tmax_stn_df['db_cluster'] == -1].shape[0]
print(f"Tmax:: Number of noise stations: {tmax_noise_count}")

# For Tmin
tmin_coords = np.radians(tmin_stn_df[['lat', 'long']].to_numpy())
tmin_db = DBSCAN(eps = epsilon, min_samples=min_samples_per_cluster, algorithm='ball_tree', metric = 'haversine').fit(tmin_coords)
tmin_stn_df['db_cluster'] = tmin_db.labels_
tmin_noise_count = tmin_stn_df[tmin_stn_df['db_cluster'] == -1].shape[0]
print(f"Tmin:: Number of noise stations: {tmin_noise_count}")

#assign db cluster values to noise stations - tmax
tmax_current_max_label = tmax_stn_df['db_cluster'].max()
tmax_start_label = tmax_current_max_label + 1
tmax_end_label = tmax_start_label + tmax_noise_count 
tmax_available_labels = np.arange(tmax_start_label, tmax_end_label)
tmax_unique_noise_labels = np.random.choice(tmax_available_labels, size=tmax_noise_count, replace=False)
tmax_stn_df.loc[tmax_stn_df['db_cluster'] == -1, 'db_cluster'] = tmax_unique_noise_labels

#assign db cluster values to noise stations - tmin
tmin_current_max_label = tmin_stn_df['db_cluster'].max()
tmin_start_label = tmin_current_max_label + 1
tmin_end_label = tmin_start_label + tmin_noise_count 
tmin_available_labels = np.arange(tmin_start_label, tmin_end_label)
tmin_unique_noise_labels = np.random.choice(tmin_available_labels, size=tmin_noise_count, replace=False)
tmin_stn_df.loc[tmin_stn_df['db_cluster'] == -1, 'db_cluster'] = tmin_unique_noise_labels

print("TMAX Head:\n", tmax_stn_df.head(3))
print("TMIN Head:\n", tmin_stn_df.head(3))

Tmax:: Number of noise stations: 31
Tmin:: Number of noise stations: 31
TMAX Head:
   index_no       lat      long  cluster_id  db_cluster
0      104  29.30135  80.58775           3          37
1      105  28.95477  80.23049           1          10
2      203  29.26804  80.98360           3           7
TMIN Head:
   index_no       lat      long  cluster_id  db_cluster
0      104  29.30135  80.58775           3          16
1      105  28.95477  80.23049           1          35
2      203  29.26804  80.98360           3          33


In [28]:
# Split stations within each cluster into train and test

# Tmax
tmax_cluster_ids = np.unique(tmax_cluster_values)
tmax_train_df = pd.DataFrame()
tmax_test_df = pd.DataFrame()
split_fraction = 0.7

for tmax_cluster_id in tmax_cluster_ids:
    tmax_cluster_stations = tmax_stn_df[tmax_stn_df['cluster_id'] == tmax_cluster_id].set_index('db_cluster')
    tmax_unique_db_clusters = tmax_cluster_stations.index.unique()
    tmax_train_clusters = pd.Series(tmax_unique_db_clusters).sample(frac=split_fraction, random_state = 64)
    
    tmax_train_df_lv = tmax_cluster_stations.loc[tmax_train_clusters].reset_index()
    tmax_test_df_lv = tmax_cluster_stations.drop(tmax_train_clusters).reset_index()
    
    tmax_train_df = pd.concat([tmax_train_df, tmax_train_df_lv], ignore_index=True)
    tmax_test_df = pd.concat([tmax_test_df, tmax_test_df_lv], ignore_index=True)

# Tmin
tmin_cluster_ids = np.unique(tmin_cluster_values)
tmin_train_df = pd.DataFrame()
tmin_test_df = pd.DataFrame()
split_fraction = 0.7

for tmin_cluster_id in tmin_cluster_ids:
    tmin_cluster_stations = tmin_stn_df[tmin_stn_df['cluster_id'] == tmin_cluster_id].set_index('db_cluster')
    tmin_unique_db_clusters = tmin_cluster_stations.index.unique()
    tmin_train_clusters = pd.Series(tmin_unique_db_clusters).sample(frac=split_fraction, random_state = 64)
    
    tmin_train_df_lv = tmin_cluster_stations.loc[tmin_train_clusters].reset_index()
    tmin_test_df_lv = tmin_cluster_stations.drop(tmin_train_clusters).reset_index()
    
    tmin_train_df = pd.concat([tmin_train_df, tmin_train_df_lv], ignore_index=True)
    tmin_test_df = pd.concat([tmin_test_df, tmin_test_df_lv], ignore_index=True)


# Save splits
save_dir = r"../Data_Temp/Processed/Split_stations/"
os.makedirs(save_dir, exist_ok=True)

tmax_train_df.to_csv(save_dir + 'tmax_train_stations.csv', index=False)
tmin_train_df.to_csv(save_dir + 'tmin_train_stations.csv', index=False)
tmax_test_df.to_csv(save_dir + 'tmax_test_stations.csv', index=False)
tmin_test_df.to_csv(save_dir + 'tmin_test_stations.csv', index=False)